## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
import xgboost as xgb

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from scipy import stats

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
plt.style.use('default')
sns.set_palette("husl")

## Data Loading

In [ ]:
def initialize_data_pipeline():
    import os
    
    if os.path.exists('/kaggle/input/Cinema_Audience_Forecasting_challenge/'):
        base_path = '/kaggle/input/Cinema_Audience_Forecasting_challenge/'
        file_mappings = {
            'visits': f'{base_path}booknow_visits/booknow_visits.csv',
            'theaters': f'{base_path}booknow_theaters/booknow_theaters.csv',
            'dates': f'{base_path}date_info/date_info.csv',
            'template': f'{base_path}sample_submission/sample_submission.csv'
        }
    else:
        base_path = r'c:\Users\priya\OneDrive\Desktop\MLP_Project'
        file_mappings = {
            'visits': os.path.join(base_path, 'booknow_visits.csv'),
            'theaters': os.path.join(base_path, 'booknow_theaters.csv'),
            'dates': os.path.join(base_path, 'date_info.csv'),
            'template': os.path.join(base_path, 'sample_submission.csv')
        }
    
    data_collection = {}
    for key, filepath in file_mappings.items():
        data_collection[key] = pd.read_csv(filepath)
        print(f"Loaded {key}: {data_collection[key].shape}")
    
    return data_collection

In [ ]:
data_store = initialize_data_pipeline()

## Exploratory Data Analysis

In [ ]:
eda_df = data_store['visits'].copy()
eda_df['show_date'] = pd.to_datetime(eda_df['show_date'])

In [ ]:
dates_df = data_store['dates'].copy()
dates_df['show_date'] = pd.to_datetime(dates_df['show_date'])

In [ ]:
eda_df = eda_df.merge(dates_df, on='show_date', how='left')
eda_df = eda_df.merge(data_store['theaters'], on='book_theater_id', how='left')

In [ ]:
eda_df['year'] = eda_df['show_date'].dt.year
eda_df['month'] = eda_df['show_date'].dt.month

In [ ]:
print(f"Dataset for EDA: {eda_df.shape}")
print(f"Date range: {eda_df['show_date'].min()} to {eda_df['show_date'].max()}")

### Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
axes[0].hist(eda_df['audience_count'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(eda_df['audience_count'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {eda_df["audience_count"].mean():.1f}')
axes[0].axvline(eda_df['audience_count'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {eda_df["audience_count"].median():.1f}')
axes[0].set_xlabel('Audience Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Audience Count', fontweight='bold', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)

In [ ]:
axes[1].boxplot(eda_df['audience_count'], vert=True)
axes[1].set_ylabel('Audience Count')
axes[1].set_title('Audience Count Box Plot', fontweight='bold', fontsize=12)
axes[1].grid(alpha=0.3)

In [ ]:
plt.tight_layout()
plt.show()

### Day of Week Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_stats = eda_df.groupby('day_of_week')['audience_count'].mean().reindex(day_order)

In [ ]:
axes[0].bar(dow_stats.index, dow_stats.values, edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Average Audience Count')
axes[0].set_title('Average Audience by Day of Week', fontweight='bold', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3, axis='y')

In [ ]:
eda_df.boxplot(column='audience_count', by='day_of_week', ax=axes[1])
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Audience Count')
axes[1].set_title('Audience Distribution by Day of Week', fontweight='bold', fontsize=12)
axes[1].get_figure().suptitle('')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
axes[1].grid(alpha=0.3)

In [ ]:
plt.tight_layout()
plt.show()

### Monthly Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
monthly_avg = eda_df.groupby(eda_df['show_date'].dt.to_period('M'))['audience_count'].mean()
monthly_avg.index = monthly_avg.index.to_timestamp()

In [ ]:
axes[0].plot(monthly_avg.index, monthly_avg.values, marker='o', linewidth=2, markersize=6)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Audience Count')
axes[0].set_title('Monthly Average Audience Trend', fontweight='bold', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3)

In [ ]:
monthly_total = eda_df.groupby(eda_df['show_date'].dt.to_period('M'))['audience_count'].sum()
monthly_total.index = monthly_total.index.to_timestamp()

In [ ]:
axes[1].bar(monthly_total.index, monthly_total.values, edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Total Audience Count')
axes[1].set_title('Monthly Total Audience Volume', fontweight='bold', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3, axis='y')

In [ ]:
plt.tight_layout()
plt.show()

### Theater Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
theater_type_dist = eda_df['theater_type'].value_counts()
axes[0].pie(theater_type_dist.values, labels=theater_type_dist.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Theater Type Distribution', fontweight='bold', fontsize=12)

In [ ]:
theater_type_avg = eda_df.groupby('theater_type')['audience_count'].mean().sort_values()
axes[1].barh(theater_type_avg.index, theater_type_avg.values, edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Average Audience Count')
axes[1].set_ylabel('Theater Type')
axes[1].set_title('Average Audience by Theater Type', fontweight='bold', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

In [ ]:
plt.tight_layout()
plt.show()

### Geographic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
top_areas_count = eda_df['theater_area'].value_counts().head(15)
axes[0].barh(range(len(top_areas_count)), top_areas_count.values, edgecolor='black', alpha=0.8)
axes[0].set_yticks(range(len(top_areas_count)))
axes[0].set_yticklabels(top_areas_count.index)
axes[0].set_xlabel('Number of Records')
axes[0].set_ylabel('Theater Area')
axes[0].set_title('Top 15 Theater Areas by Volume', fontweight='bold', fontsize=12)
axes[0].grid(alpha=0.3, axis='x')

In [ ]:
top_areas_avg = eda_df.groupby('theater_area')['audience_count'].mean().sort_values(ascending=False).head(15)
axes[1].barh(range(len(top_areas_avg)), top_areas_avg.values, edgecolor='black', alpha=0.8, color='coral')
axes[1].set_yticks(range(len(top_areas_avg)))
axes[1].set_yticklabels(top_areas_avg.index)
axes[1].set_xlabel('Average Audience Count')
axes[1].set_ylabel('Theater Area')
axes[1].set_title('Top 15 Areas by Average Audience', fontweight='bold', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

In [ ]:
plt.tight_layout()
plt.show()

### Theater Performance Rankings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
theater_performance = eda_df.groupby('book_theater_id')['audience_count'].mean().sort_values()

In [ ]:
top_15 = theater_performance.tail(15)
axes[0].barh(range(len(top_15)), top_15.values, edgecolor='black', alpha=0.8, color='green')
axes[0].set_yticks(range(len(top_15)))
axes[0].set_yticklabels(top_15.index)
axes[0].set_xlabel('Average Audience Count')
axes[0].set_ylabel('Theater ID')
axes[0].set_title('Top 15 Performing Theaters', fontweight='bold', fontsize=12)
axes[0].grid(alpha=0.3, axis='x')

In [ ]:
bottom_15 = theater_performance.head(15)
axes[1].barh(range(len(bottom_15)), bottom_15.values, edgecolor='black', alpha=0.8, color='red')
axes[1].set_yticks(range(len(bottom_15)))
axes[1].set_yticklabels(bottom_15.index)
axes[1].set_xlabel('Average Audience Count')
axes[1].set_ylabel('Theater ID')
axes[1].set_title('Bottom 15 Performing Theaters', fontweight='bold', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

In [ ]:
plt.tight_layout()
plt.show()

### Weekend vs Weekday Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
eda_df['is_weekend'] = eda_df['show_date'].dt.dayofweek.isin([5, 6]).astype(int)
weekend_comparison = eda_df.groupby('is_weekend')['audience_count'].mean()
weekend_labels = ['Weekday', 'Weekend']

In [ ]:
axes[0].bar(weekend_labels, weekend_comparison.values, edgecolor='black', alpha=0.8, color=['lightblue', 'lightcoral'])
axes[0].set_ylabel('Average Audience Count')
axes[0].set_title('Average Audience: Weekday vs Weekend', fontweight='bold', fontsize=12)
axes[0].grid(alpha=0.3, axis='y')

for i, v in enumerate(weekend_comparison.values):
    axes[0].text(i, v + 1, f'{v:.1f}', ha='center', fontweight='bold')

In [ ]:
weekday_data = eda_df[eda_df['is_weekend'] == 0]['audience_count']
weekend_data = eda_df[eda_df['is_weekend'] == 1]['audience_count']

In [ ]:
axes[1].boxplot([weekday_data, weekend_data], labels=weekend_labels)
axes[1].set_ylabel('Audience Count')
axes[1].set_title('Audience Distribution: Weekday vs Weekend', fontweight='bold', fontsize=12)
axes[1].grid(alpha=0.3, axis='y')

In [ ]:
plt.tight_layout()
plt.show()

### Correlation Analysis

In [ ]:
numeric_cols = eda_df.select_dtypes(include=[np.number]).columns
correlation_matrix = eda_df[numeric_cols].corr()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
target_corr = correlation_matrix['audience_count'].sort_values(ascending=False)
print("\nTop 10 Features Correlated with Audience Count:")
print(target_corr.head(11)[1:])

### Year-over-Year Analysis

In [ ]:
yearly_stats = eda_df.groupby('year')['audience_count'].agg(['mean', 'median', 'std', 'count'])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

In [ ]:
yearly_stats['mean'].plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Average Audience by Year', fontweight='bold')
axes[0, 0].set_ylabel('Average Audience')
axes[0, 0].grid(True, alpha=0.3)

In [ ]:
yearly_stats['count'].plot(kind='bar', ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_title('Number of Shows by Year', fontweight='bold')
axes[0, 1].set_ylabel('Number of Shows')
axes[0, 1].grid(True, alpha=0.3)

In [ ]:
yearly_stats['median'].plot(kind='bar', ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Median Audience by Year', fontweight='bold')
axes[1, 0].set_ylabel('Median Audience')
axes[1, 0].grid(True, alpha=0.3)

In [ ]:
yearly_stats['std'].plot(kind='bar', ax=axes[1, 1], color='plum')
axes[1, 1].set_title('Audience Variability by Year', fontweight='bold')
axes[1, 1].set_ylabel('Standard Deviation')
axes[1, 1].grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.show()

### Statistical Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

In [ ]:
axes[0, 0].hist(eda_df['audience_count'], bins=50, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Original Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Audience Count')
axes[0, 0].set_ylabel('Frequency')

In [ ]:
stats.probplot(eda_df['audience_count'], dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Normal)', fontweight='bold')

In [ ]:
log_audience = np.log1p(eda_df['audience_count'])
axes[1, 0].hist(log_audience, bins=50, color='lightcoral', edgecolor='black')
axes[1, 0].set_title('Log-Transformed Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Log(Audience Count + 1)')
axes[1, 0].set_ylabel('Frequency')

In [ ]:
stats.probplot(log_audience, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot Log-Transformed', fontweight='bold')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
print(f"Skewness (Original): {eda_df['audience_count'].skew():.2f}")
print(f"Skewness (Log-transformed): {log_audience.skew():.2f}")

### Monthly Patterns

In [ ]:
monthly_stats = eda_df.groupby('month')['audience_count'].agg(['mean', 'median', 'count'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

In [ ]:
monthly_stats['mean'].plot(kind='line', marker='o', ax=axes[0], color='blue')
axes[0].set_title('Average Audience by Month', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Audience')
axes[0].grid(True, alpha=0.3)

In [ ]:
monthly_stats['median'].plot(kind='line', marker='s', ax=axes[1], color='green')
axes[1].set_title('Median Audience by Month', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Median Audience')
axes[1].grid(True, alpha=0.3)

In [ ]:
monthly_stats['count'].plot(kind='bar', ax=axes[2], color='coral')
axes[2].set_title('Number of Shows by Month', fontweight='bold')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Number of Shows')
axes[2].grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.show()

### Data Quality Check

In [ ]:
print("Dataset Shape:", eda_df.shape)
print("\nMissing Values:")
print(eda_df.isnull().sum())

In [ ]:
print("\nData Types:")
print(eda_df.dtypes)

In [ ]:
print("\nBasic Statistics:")
print(eda_df.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
missing_data = eda_df.isnull().sum()
if missing_data.sum() > 0:
    missing_data[missing_data > 0].plot(kind='bar', ax=axes[0], color='red')
    axes[0].set_title('Missing Values by Column', fontweight='bold')
    axes[0].set_ylabel('Count')
else:
    axes[0].text(0.5, 0.5, 'No Missing Values!', ha='center', va='center', fontsize=20, fontweight='bold', color='green')
    axes[0].set_title('Missing Values Status', fontweight='bold')

In [ ]:
duplicate_count = eda_df.duplicated().sum()
labels = ['Unique Records', 'Duplicates']
sizes = [len(eda_df) - duplicate_count, duplicate_count]
colors = ['lightblue', 'lightcoral']
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Duplicate Records', fontweight='bold')

In [ ]:
plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
def build_feature_matrix(data_store):
    primary_df = data_store['visits'].copy()
    primary_df['show_date'] = pd.to_datetime(primary_df['show_date'])
    
    date_features = data_store['dates'].copy()
    date_features['show_date'] = pd.to_datetime(date_features['show_date'])
    primary_df = primary_df.merge(date_features, on='show_date', how='left')
    
    primary_df = primary_df.merge(data_store['theaters'], on='book_theater_id', how='left')
    
    return primary_df

In [ ]:
enhanced_df = build_feature_matrix(data_store)
print(f"Initial feature matrix: {enhanced_df.shape}")

In [ ]:
def handle_missing_values(dataframe):
    categorical_cols = ['theater_type', 'theater_area']
    numerical_cols = ['latitude', 'longitude']
    
    for col in categorical_cols:
        if col in dataframe.columns:
            mode_value = dataframe[col].mode()[0]
            dataframe[col].fillna(mode_value, inplace=True)
    
    for col in numerical_cols:
        if col in dataframe.columns:
            median_value = dataframe[col].median()
            dataframe[col].fillna(median_value, inplace=True)
    
    return dataframe

In [ ]:
enhanced_df = handle_missing_values(enhanced_df)
print(f"Missing values after imputation: {enhanced_df.isnull().sum().sum()}")

In [ ]:
def extract_temporal_patterns(dataframe):
    dataframe['month_num'] = dataframe['show_date'].dt.month
    dataframe['day_of_week'] = dataframe['show_date'].dt.dayofweek
    dataframe['weekend_flag'] = (dataframe['day_of_week'] >= 5).astype(int)
    
    dataframe['month_sine'] = np.sin(2 * np.pi * dataframe['month_num'] / 12)
    dataframe['month_cosine'] = np.cos(2 * np.pi * dataframe['month_num'] / 12)
    dataframe['weekday_sine'] = np.sin(2 * np.pi * dataframe['day_of_week'] / 7)
    dataframe['weekday_cosine'] = np.cos(2 * np.pi * dataframe['day_of_week'] / 7)
    
    return dataframe

In [ ]:
enhanced_df = extract_temporal_patterns(enhanced_df)
print("Temporal features created")

In [ ]:
def create_lag_features(dataframe):
    dataframe = dataframe.sort_values(['book_theater_id', 'show_date'])
    
    lag_periods = [1, 2, 7, 14]
    for period in lag_periods:
        col_name = f'lag_audience_{period}d'
        dataframe[col_name] = dataframe.groupby('book_theater_id')['audience_count'].shift(period)
    
    return dataframe

In [ ]:
enhanced_df = create_lag_features(enhanced_df)
print("Lag features generated")

In [ ]:
def create_rolling_statistics(dataframe):
    window_sizes = [7, 14]
    
    for window in window_sizes:
        mean_col = f'rolling_avg_{window}d'
        dataframe[mean_col] = dataframe.groupby('book_theater_id')['audience_count'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
        )
    
    return dataframe

In [ ]:
enhanced_df = create_rolling_statistics(enhanced_df)
print("Rolling statistics computed")

In [ ]:
def compute_theater_metrics(dataframe):
    cutoff_date = dataframe['show_date'].max() - pd.Timedelta(days=365)
    historical_subset = dataframe[dataframe['show_date'] <= cutoff_date]
    
    theater_statistics = historical_subset.groupby('book_theater_id')['audience_count'].agg(
        ['mean', 'std', 'median']
    ).reset_index()
    theater_statistics.columns = ['book_theater_id', 'avg_audience', 'std_audience', 'median_audience']
    
    dataframe = dataframe.merge(theater_statistics, on='book_theater_id', how='left')
    
    global_avg = historical_subset['audience_count'].mean()
    global_std = historical_subset['audience_count'].std()
    global_median = historical_subset['audience_count'].median()
    
    dataframe['avg_audience'].fillna(global_avg, inplace=True)
    dataframe['std_audience'].fillna(global_std, inplace=True)
    dataframe['median_audience'].fillna(global_median, inplace=True)
    
    return dataframe

In [ ]:
enhanced_df = compute_theater_metrics(enhanced_df)
print("Theater metrics calculated")

In [ ]:
def generate_interaction_features(dataframe):
    dataframe['weekend_x_type'] = dataframe['weekend_flag'].astype(str) + '_' + dataframe['theater_type']
    dataframe['month_x_type'] = dataframe['month_num'].astype(str) + '_' + dataframe['theater_type']
    
    return dataframe

In [ ]:
enhanced_df = generate_interaction_features(enhanced_df)
print("Interaction features created")

In [ ]:
encoder_area = LabelEncoder()
encoder_theater = LabelEncoder()

In [ ]:
enhanced_df['area_encoded'] = encoder_area.fit_transform(enhanced_df['theater_area'])
enhanced_df['theater_id_encoded'] = encoder_theater.fit_transform(enhanced_df['book_theater_id'])

In [ ]:
type_dummies = pd.get_dummies(enhanced_df['theater_type'], prefix='type')
enhanced_df = pd.concat([enhanced_df, type_dummies], axis=1)

In [ ]:
print(f"Final feature count: {enhanced_df.shape[1]}")

## Data Preparation

In [ ]:
def prepare_model_inputs(dataframe):
    excluded_columns = [
        'show_date', 'book_theater_id', 'theater_type', 'theater_area',
        'day_of_week', 'audience_count', 'year', 'day', 'quarter', 'day_of_year',
        'latitude', 'longitude', 'weekend_x_type', 'month_x_type'
    ]
    
    feature_list = [col for col in dataframe.columns if col not in excluded_columns]
    
    X_matrix = dataframe[feature_list].fillna(0)
    y_target = dataframe['audience_count']
    
    print(f"Feature dimensions: {X_matrix.shape}")
    print(f"Target samples: {len(y_target)}")
    
    return X_matrix, y_target, feature_list

In [ ]:
X_features, y_target, selected_features = prepare_model_inputs(enhanced_df)

In [ ]:
def split_temporal_data(X_matrix, y_target, dataframe):
    validation_cutoff = dataframe['show_date'].max() - pd.Timedelta(days=365)
    
    train_indices = dataframe['show_date'] <= validation_cutoff
    test_indices = dataframe['show_date'] > validation_cutoff
    
    X_train = X_matrix[train_indices]
    X_val = X_matrix[test_indices]
    y_train = y_target[train_indices]
    y_val = y_target[test_indices]
    
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Validation samples: {X_val.shape[0]}")
    print(f"Split date: {validation_cutoff.date()}")
    
    return X_train, X_val, y_train, y_val

In [ ]:
X_train, X_val, y_train, y_val = split_temporal_data(X_features, y_target, enhanced_df)

## Model Training

In [ ]:
feature_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)

In [ ]:
print("Features standardized")

### Ridge Regression

In [ ]:
ridge_param_grid = {
    'alpha': [1.0, 2.0, 3.0, 5.0, 10.0]
}

In [ ]:
ridge_base = Ridge(random_state=42)
ridge_grid_search = GridSearchCV(
    ridge_base,
    ridge_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

In [ ]:
print("Starting Ridge hyperparameter tuning...")
ridge_grid_search.fit(X_train_scaled, y_train)

In [ ]:
best_ridge = ridge_grid_search.best_estimator_
print(f"Best Ridge parameters: {ridge_grid_search.best_params_}")
print(f"Best CV score: {ridge_grid_search.best_score_:.4f}")

In [ ]:
ridge_predictions = best_ridge.predict(X_val_scaled)
ridge_score = r2_score(y_val, ridge_predictions)
print(f"Ridge validation R²: {ridge_score:.4f}")

### Random Forest

In [ ]:
rf_param_grid = {
    'n_estimators': [150, 200, 250],
    'max_depth': [12, 15, 18],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2]
}

In [ ]:
rf_base = RandomForestRegressor(
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf_grid_search = GridSearchCV(
    rf_base,
    rf_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

In [ ]:
print("Starting Random Forest hyperparameter tuning...")
rf_grid_search.fit(X_train, y_train)

In [ ]:
best_rf = rf_grid_search.best_estimator_
print(f"Best Random Forest parameters: {rf_grid_search.best_params_}")
print(f"Best CV score: {rf_grid_search.best_score_:.4f}")

In [ ]:
rf_predictions = best_rf.predict(X_val)
rf_score = r2_score(y_val, rf_predictions)
print(f"Random Forest validation R²: {rf_score:.4f}")

### XGBoost

In [ ]:
xgb_param_grid = {
    'n_estimators': [150, 200, 250],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.05, 0.06, 0.08],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9]
}

In [ ]:
xgb_base = xgb.XGBRegressor(
    reg_alpha=0.05,
    reg_lambda=0.05,
    random_state=42,
    n_jobs=-1
)

In [ ]:
xgb_grid_search = GridSearchCV(
    xgb_base,
    xgb_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

In [ ]:
print("Starting XGBoost hyperparameter tuning...")
xgb_grid_search.fit(X_train, y_train)

In [ ]:
best_xgb = xgb_grid_search.best_estimator_
print(f"Best XGBoost parameters: {xgb_grid_search.best_params_}")
print(f"Best CV score: {xgb_grid_search.best_score_:.4f}")

In [ ]:
xgb_predictions = best_xgb.predict(X_val)
xgb_score = r2_score(y_val, xgb_predictions)
print(f"XGBoost validation R²: {xgb_score:.4f}")

## Ensemble Model

In [ ]:
model_scores = np.array([ridge_score, rf_score, xgb_score])
model_predictions = np.array([ridge_predictions, rf_predictions, xgb_predictions])

In [ ]:
exponential_weights = np.exp(model_scores * 2)
normalized_weights = exponential_weights / np.sum(exponential_weights)

In [ ]:
print(f"Individual scores: {model_scores}")
print(f"Ensemble weights: {normalized_weights}")

In [ ]:
ensemble_predictions = np.average(model_predictions, axis=0, weights=normalized_weights)

In [ ]:
ensemble_r2 = r2_score(y_val, ensemble_predictions)
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_predictions))
ensemble_mae = mean_absolute_error(y_val, ensemble_predictions)

In [ ]:
print(f"\nEnsemble Performance:")
print(f"R² Score: {ensemble_r2:.4f}")
print(f"RMSE: {ensemble_rmse:.2f}")
print(f"MAE: {ensemble_mae:.2f}")

## Test Data Processing

In [ ]:
def prepare_test_dataset(data_store, training_df, encoders):
    template_df = data_store['template'].copy()
    
    template_df['book_theater_id'] = template_df['ID'].str.split('_').str[0] + '_' + template_df['ID'].str.split('_').str[1]
    template_df['show_date'] = pd.to_datetime(template_df['ID'].str.split('_').str[2])
    
    test_df = template_df[['book_theater_id', 'show_date']].copy()
    test_df = test_df.merge(data_store['theaters'], on='book_theater_id', how='left')
    
    date_features = data_store['dates'].copy()
    date_features['show_date'] = pd.to_datetime(date_features['show_date'])
    test_df = test_df.merge(date_features, on='show_date', how='left')
    
    test_df = handle_missing_values(test_df)
    
    return test_df

In [ ]:
test_dataset = prepare_test_dataset(data_store, enhanced_df, {
    'encoder_area': encoder_area,
    'encoder_theater': encoder_theater
})

In [ ]:
print(f"Test dataset shape: {test_dataset.shape}")

In [ ]:
def engineer_test_features(test_df, training_df, encoders):
    test_df = extract_temporal_patterns(test_df)
    
    theater_avg_audience = training_df.groupby('book_theater_id')['audience_count'].mean()
    default_audience = training_df['audience_count'].mean()
    
    lag_periods = [1, 2, 7, 14]
    for period in lag_periods:
        col_name = f'lag_audience_{period}d'
        test_df[col_name] = test_df['book_theater_id'].map(theater_avg_audience).fillna(default_audience)
    
    window_sizes = [7, 14]
    for window in window_sizes:
        col_name = f'rolling_avg_{window}d'
        test_df[col_name] = test_df['lag_audience_1d']
    
    theater_stats = training_df.groupby('book_theater_id')['audience_count'].agg(
        ['mean', 'std', 'median']
    ).reset_index()
    theater_stats.columns = ['book_theater_id', 'avg_audience', 'std_audience', 'median_audience']
    
    test_df = test_df.merge(theater_stats, on='book_theater_id', how='left')
    
    test_df['avg_audience'].fillna(training_df['audience_count'].mean(), inplace=True)
    test_df['std_audience'].fillna(training_df['audience_count'].std(), inplace=True)
    test_df['median_audience'].fillna(training_df['audience_count'].median(), inplace=True)
    
    test_df = generate_interaction_features(test_df)
    
    area_mapping = dict(zip(encoders['encoder_area'].classes_, 
                           encoders['encoder_area'].transform(encoders['encoder_area'].classes_)))
    test_df['area_encoded'] = test_df['theater_area'].map(area_mapping).fillna(-1)
    
    theater_mapping = dict(zip(encoders['encoder_theater'].classes_,
                              encoders['encoder_theater'].transform(encoders['encoder_theater'].classes_)))
    test_df['theater_id_encoded'] = test_df['book_theater_id'].map(theater_mapping).fillna(-1)
    
    type_dummies = pd.get_dummies(test_df['theater_type'], prefix='type')
    
    for col in encoders['type_columns']:
        if col not in type_dummies.columns:
            type_dummies[col] = 0
    
    type_dummies = type_dummies[encoders['type_columns']]
    test_df = pd.concat([test_df, type_dummies], axis=1)
    
    return test_df

In [ ]:
type_columns = [col for col in selected_features if 'type_' in col]

In [ ]:
test_features = engineer_test_features(test_dataset, enhanced_df, {
    'encoder_area': encoder_area,
    'encoder_theater': encoder_theater,
    'type_columns': type_columns
})

In [ ]:
print(f"Test features engineered: {test_features.shape}")

## Generate Predictions

In [ ]:
test_X = test_features[selected_features].fillna(0)
test_X_scaled = feature_scaler.transform(test_X)

In [ ]:
print(f"Test feature matrix: {test_X.shape}")
print(f"Missing values: {test_X.isnull().sum().sum()}")

In [ ]:
ridge_test_pred = best_ridge.predict(test_X_scaled)
rf_test_pred = best_rf.predict(test_X)
xgb_test_pred = best_xgb.predict(test_X)

In [ ]:
final_predictions = np.average(
    [ridge_test_pred, rf_test_pred, xgb_test_pred],
    axis=0,
    weights=normalized_weights
)

In [ ]:
final_predictions = np.maximum(final_predictions, 0)

In [ ]:
training_95th = np.percentile(enhanced_df['audience_count'], 95)
final_predictions = np.minimum(final_predictions, training_95th)

In [ ]:
final_predictions = np.round(final_predictions).astype(int)

In [ ]:
print(f"\nPrediction Summary:")
print(f"Count: {len(final_predictions)}")
print(f"Min: {final_predictions.min()}")
print(f"Max: {final_predictions.max()}")
print(f"Mean: {final_predictions.mean():.1f}")
print(f"Median: {np.median(final_predictions):.1f}")

## Create Submission

In [ ]:
submission_df = pd.DataFrame({
    'ID': data_store['template']['ID'],
    'audience_count': final_predictions
})

In [ ]:
submission_df.to_csv('submission.csv', index=False)

In [ ]:
print("Submission file created: submission.csv")
print(f"Shape: {submission_df.shape}")
print("\nFirst 10 predictions:")
print(submission_df.head(10))